# Bagian B – Training SigLIP di Google Colab

**Checkpoint model:** `google/siglip-so400m-patch14-384`  
**Fokus tahap ini:** Tahap 5 – Preprocessing + DataLoader yang benar untuk SigLIP

**Prasyarat:** Sudah menjalankan `preprocessing_lokal.ipynb` dan mengunggah hasilnya ke Google Drive sesuai instruksi di akhir notebook lokal.

---
## CELL 1 – Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("[OK] Google Drive berhasil di-mount.")

---
## CELL 2 – Konfigurasi Root & Path

In [ ]:
import os

# ============================================================
# KONFIGURASI UTAMA – Sesuaikan ROOT_DIR dengan lokasi upload
# ============================================================
ROOT_DIR    = "/content/drive/MyDrive/Senpro"          # root di Google Drive
DATASET_DIR = os.path.join(ROOT_DIR, "Dataset")         # folder gambar
OUTPUT_DIR  = os.path.join(ROOT_DIR, "preprocessing_output")

# Path manifest
TRAIN_CSV   = os.path.join(OUTPUT_DIR, "train.csv")
VAL_CSV     = os.path.join(OUTPUT_DIR, "val.csv")
TEST_CSV    = os.path.join(OUTPUT_DIR, "test.csv")
CW_JSON     = os.path.join(OUTPUT_DIR, "class_weights.json")
AUG_JSON    = os.path.join(OUTPUT_DIR, "augmentasi_config.json")

# Model checkpoint
MODEL_CHECKPOINT = "google/siglip-so400m-patch14-384"

# Hyperparameter training
BATCH_SIZE     = 32          # turunkan ke 16 jika OOM
NUM_EPOCHS     = 20
LEARNING_RATE  = 2e-4
FREEZE_BACKBONE = True       # True = hanya train classification head
                              # False = fine-tune seluruh model (lebih lambat)
NUM_CLASSES    = 3
DROPOUT_RATE   = 0.3
RANDOM_SEED    = 42

# DataLoader
NUM_WORKERS    = 2           # Colab gratis: CPU terbatas, jangan terlalu tinggi
PIN_MEMORY     = True        # Akselerasi transfer CPU→GPU

# Checkpointing ke Drive
CHECKPOINT_DIR = os.path.join(ROOT_DIR, "checkpoints")
SAVE_EVERY_N_EPOCHS = 2     # simpan checkpoint setiap N epoch

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Label mapping
LABEL_TO_IDX = {"Recylable": 0, "Organik": 1, "B3": 2}
IDX_TO_LABEL = {v: k for k, v in LABEL_TO_IDX.items()}

# Validasi path
for name, path in [("ROOT_DIR", ROOT_DIR), ("DATASET_DIR", DATASET_DIR),
                   ("TRAIN_CSV", TRAIN_CSV), ("VAL_CSV", VAL_CSV),
                   ("TEST_CSV", TEST_CSV), ("CW_JSON", CW_JSON)]:
    exists = os.path.exists(path)
    status = "✓" if exists else "✗ TIDAK DITEMUKAN"
    print(f"  [{status}] {name}: {path}")

---
## CELL 3 – Cek GPU & Instalasi Dependency

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"[OK] GPU tersedia: {torch.cuda.get_device_name(0)}")
    print(f"     VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("[WARN] GPU tidak tersedia! Training akan sangat lambat di CPU.")
    print("  ➜ Cara mengaktifkan GPU di Colab:")
    print("    Menu atas: Runtime → Change runtime type → Hardware accelerator → GPU → Save")

print(f"\n[INFO] Device yang digunakan: {device}")

In [ ]:
# Install dependency yang belum tersedia default di Colab
# Jalankan cell ini sekali; restart runtime jika diminta
!pip install -q --upgrade transformers>=4.37.0
!pip install -q opencv-python-headless pillow tqdm
print("[OK] Instalasi selesai.")

---
## CELL 4 – Import & Reproducibility

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from transformers import SiglipModel, SiglipImageProcessor

# ── Reproducibility ──────────────────────────────────────────
def set_seed(seed: int):
    """Set semua random seed agar hasil reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Deterministic cudnn (sedikit lebih lambat, tapi reproducible)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(RANDOM_SEED)
print(f"[OK] Random seed dikunci: {RANDOM_SEED}")
print(f"[OK] Semua library berhasil diimport.")

---
## CELL 5 – Muat Manifest & Class Weights

In [ ]:
def load_manifest(csv_path: str, dataset_dir: str) -> pd.DataFrame:
    """
    Muat manifest CSV dan bentuk kolom 'abs_path' dengan menggabungkan
    dataset_dir + rel_path.
    Verifikasi bahwa setiap file benar-benar ada di Drive.
    """
    df = pd.read_csv(csv_path)
    df['abs_path'] = df['rel_path'].apply(
        lambda p: os.path.join(dataset_dir, p.replace('/', os.sep))
    )
    missing = df[~df['abs_path'].apply(os.path.exists)]
    if len(missing) > 0:
        print(f"  [WARN] {len(missing)} file tidak ditemukan di Drive (mungkin gagal upload).")
        # Buang baris yang filenya tidak ada agar tidak crash saat training
        df = df[df['abs_path'].apply(os.path.exists)].copy().reset_index(drop=True)
    return df


# Muat semua split
df_train = load_manifest(TRAIN_CSV, DATASET_DIR)
df_val   = load_manifest(VAL_CSV,   DATASET_DIR)
df_test  = load_manifest(TEST_CSV,  DATASET_DIR)

print("=== VERIFIKASI MANIFEST ===")
for split_name, df_s in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    print(f"\n  {split_name}: {len(df_s)} sampel")
    for label, cnt in sorted(df_s['label'].value_counts().items()):
        print(f"    {label}: {cnt}")

In [ ]:
# Muat class weights dari JSON Bagian A
with open(CW_JSON, 'r') as f:
    class_weights_dict = json.load(f)

print("=== CLASS WEIGHTS ===")
# Konversi ke tensor PyTorch (urutan sesuai LABEL_TO_IDX)
cw_tensor = torch.tensor(
    [class_weights_dict[lbl] for lbl in sorted(LABEL_TO_IDX, key=LABEL_TO_IDX.get)],
    dtype=torch.float32
).to(device)

for label, w in class_weights_dict.items():
    print(f"  {label}: {w:.4f}")
print(f"\n  Tensor weights (Recylable, Organik, B3): {cw_tensor.tolist()}")

---
## CELL 6 – Augmentasi & Dataset Class

In [ ]:
def build_augmentation_transforms(split: str, label: str = None) -> transforms.Compose:
    """
    Bangun pipeline augmentasi PyTorch sesuai split dan label.

    PENTING: Augmentasi dilakukan SEBELUM SiglipImageProcessor.
    Output fungsi ini adalah PIL Image yang sudah diaugmentasi,
    kemudian akan diproses oleh SiglipImageProcessor.

    Args:
        split: 'train', 'val', atau 'test'
        label: label kelas, dipakai untuk augmentasi ekstra khusus B3

    Returns:
        transforms.Compose untuk PIL Image → PIL Image
    """
    if split != 'train':
        # Val dan test: TIDAK ada augmentasi, hanya pastikan ukuran cukup besar
        # Resize minimal ke 384 agar SiglipImageProcessor tidak perlu upscale ekstrem
        return transforms.Compose([
            transforms.Resize((384, 384), interpolation=transforms.InterpolationMode.BICUBIC),
        ])

    # === Augmentasi TRAIN ===
    if label == 'B3':
        # Augmentasi lebih agresif untuk kelas minoritas B3
        aug = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.3),
            transforms.RandomRotation(degrees=30),
            transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1),
            transforms.RandomResizedCrop(size=(384, 384), scale=(0.7, 1.0),
                                         interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.RandomGrayscale(p=0.05),
        ])
    else:
        # Augmentasi umum untuk Recylable dan Organik
        aug = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.2),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
            transforms.RandomResizedCrop(size=(384, 384), scale=(0.8, 1.0),
                                         interpolation=transforms.InterpolationMode.BICUBIC),
        ])
    return aug


print("[OK] Fungsi augmentasi berhasil didefinisikan.")

In [ ]:
class WasteDataset(Dataset):
    """
    Dataset klasifikasi sampah untuk SigLIP.

    Pipeline per item:
    1. Buka gambar sebagai PIL Image (RGB)
    2. Terapkan augmentasi (jika split='train')
    3. Proses dengan SiglipImageProcessor → tensor pixel_values [3, 384, 384]

    CATATAN DESAIN:
    - Augmentasi dikerjakan SEBELUM processor, karena SiglipImageProcessor
      mengharapkan input PIL/numpy dan mengurus resize + normalisasi ke [-1, 1]
      sesuai checkpoint (bukan normalisasi ImageNet manual).
    - Pembacaan dilakukan on-the-fly per item → hemat RAM.
    """

    def __init__(self, df: pd.DataFrame, processor: SiglipImageProcessor,
                 split: str, label_to_idx: dict):
        """
        Args:
            df           : DataFrame dengan kolom 'abs_path' dan 'label'
            processor    : SiglipImageProcessor instance
            split        : 'train' | 'val' | 'test'
            label_to_idx : dict mapping label string ke integer
        """
        self.df           = df.reset_index(drop=True)
        self.processor    = processor
        self.split        = split
        self.label_to_idx = label_to_idx

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict:
        row      = self.df.iloc[idx]
        img_path = row['abs_path']
        label    = row['label']

        # 1. Buka gambar
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            # Fallback: buat gambar hitam jika gagal buka
            print(f"[WARN] Gagal membuka {img_path}: {e}")
            image = Image.new('RGB', (384, 384), color=0)

        # 2. Augmentasi (sebelum processor, hanya untuk split='train')
        aug_transform = build_augmentation_transforms(self.split, label=label)
        image = aug_transform(image)

        # 3. SiglipImageProcessor – resize ke 384x384 dan normalize ke [-1, 1]
        # do_rescale=True, do_normalize=True secara default sesuai checkpoint
        # JANGAN normalisasi manual dengan mean/std ImageNet
        processed = self.processor(
            images=image,
            return_tensors='pt'
        )
        # processed['pixel_values'] shape: [1, 3, 384, 384] → squeeze ke [3, 384, 384]
        pixel_values = processed['pixel_values'].squeeze(0)

        # 4. Label → integer
        label_idx = self.label_to_idx[label]

        return {
            'pixel_values': pixel_values,
            'labels': torch.tensor(label_idx, dtype=torch.long),
            'path': img_path,
        }


print("[OK] WasteDataset berhasil didefinisikan.")

---
## CELL 7 – Load SiglipImageProcessor & Buat DataLoader

In [ ]:
print(f"[INFO] Memuat SiglipImageProcessor dari: {MODEL_CHECKPOINT}")
print("  (Pertama kali akan mendownload config ~beberapa MB)")

processor = SiglipImageProcessor.from_pretrained(MODEL_CHECKPOINT)

print("[OK] SiglipImageProcessor berhasil dimuat.")
print(f"  Image size yang diharapkan: {processor.size}")
print(f"  Rescale factor            : {processor.rescale_factor}")
print(f"  Image mean                : {processor.image_mean}")
print(f"  Image std                 : {processor.image_std}")

In [ ]:
# Buat Dataset instances
train_dataset = WasteDataset(df_train, processor, split='train', label_to_idx=LABEL_TO_IDX)
val_dataset   = WasteDataset(df_val,   processor, split='val',   label_to_idx=LABEL_TO_IDX)
test_dataset  = WasteDataset(df_test,  processor, split='test',  label_to_idx=LABEL_TO_IDX)

# Buat DataLoader
# persistent_workers=True agar worker tidak di-respawn setiap epoch (lebih cepat)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)

print("=== DATALOADER SIAP ===")
print(f"  Train: {len(train_dataset)} sampel, {len(train_loader)} batch")
print(f"  Val  : {len(val_dataset)} sampel, {len(val_loader)} batch")
print(f"  Test : {len(test_dataset)} sampel, {len(test_loader)} batch")

In [ ]:
# Verifikasi satu batch: cek shape tensor
sample_batch = next(iter(train_loader))
print("=== VERIFIKASI BATCH PERTAMA ===")
print(f"  pixel_values shape: {sample_batch['pixel_values'].shape}")
print(f"    → Harusnya: [batch_size, 3, 384, 384]")
print(f"  labels shape      : {sample_batch['labels'].shape}")
print(f"  pixel_values range: [{sample_batch['pixel_values'].min():.2f}, {sample_batch['pixel_values'].max():.2f}]")
print(f"    → SigLIP expects [-1, 1], pastikan di range ini")
print(f"  Label contoh      : {[IDX_TO_LABEL[l.item()] for l in sample_batch['labels'][:4]]}")

---
## CELL 8 – Model: SigLIP Backbone + Classification Head

In [ ]:
class SigLIPClassifier(nn.Module):
    """
    Classifier berbasis SigLIP vision encoder + classification head.

    DESAIN:
    - Backbone: SiglipModel vision encoder (so400m-patch14-384)
      Output: pooled embedding [batch, 1152] dari CLS token / mean pooling
    - Head: Linear → GELU → Dropout → Linear → 3 kelas

    FREEZE:
    - freeze_backbone=True  → hanya train head (cepat, bagus untuk dataset kecil)
    - freeze_backbone=False → fine-tune semua parameter (butuh GPU besar & LR kecil)
    """

    def __init__(self, model_checkpoint: str, num_classes: int,
                 dropout: float = 0.3, freeze_backbone: bool = True):
        super().__init__()

        # Muat hanya vision encoder dari SiglipModel
        # Kita tidak butuh text encoder untuk klasifikasi
        siglip = SiglipModel.from_pretrained(model_checkpoint)
        self.vision_model = siglip.vision_model

        if freeze_backbone:
            # Bekukan semua parameter backbone
            for param in self.vision_model.parameters():
                param.requires_grad = False
            print("[INFO] Backbone SigLIP dibekukan (hanya head yang di-train).")
        else:
            print("[INFO] Seluruh model akan di-fine-tune (backbone + head).")

        # Deteksi dimensi embedding output vision encoder
        # so400m → 1152
        embed_dim = self.vision_model.config.hidden_size
        print(f"[INFO] Dimensi embedding backbone: {embed_dim}")

        # Classification head: MLP sederhana
        hidden_dim = embed_dim // 2  # 576 untuk so400m
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

        # Inisialisasi head dengan Xavier
        for layer in self.classifier:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        Args:
            pixel_values: tensor [batch, 3, 384, 384] dari SiglipImageProcessor
        Returns:
            logits: tensor [batch, num_classes]
        """
        # Vision encoder → output berupa BaseModelOutputWithPooling
        vision_out = self.vision_model(pixel_values=pixel_values)

        # Gunakan pooler_output (representasi global gambar)
        # Shape: [batch, embed_dim]
        pooled = vision_out.pooler_output

        # Classification head
        logits = self.classifier(pooled)
        return logits


print("[OK] SigLIPClassifier berhasil didefinisikan.")

In [ ]:
print(f"[INFO] Memuat SigLIP backbone dari HuggingFace: {MODEL_CHECKPOINT}")
print("  (Download pertama ~700 MB, selanjutnya dari cache)")

model = SigLIPClassifier(
    model_checkpoint=MODEL_CHECKPOINT,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT_RATE,
    freeze_backbone=FREEZE_BACKBONE,
).to(device)

# Ringkasan parameter
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n  Total parameter    : {total_params:,}")
print(f"  Parameter trainable: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")

---
## CELL 9 – Loss Function & Optimizer

In [ ]:
# ============================================================
# PILIHAN LOSS FUNCTION
# Ubah USE_FOCAL_LOSS = True jika ingin mencoba Focal Loss
# ============================================================
USE_FOCAL_LOSS = False   # False = CrossEntropyLoss dengan class weights
FOCAL_GAMMA    = 2.0     # gamma untuk focal loss (hanya dipakai jika USE_FOCAL_LOSS=True)


class FocalLoss(nn.Module):
    """
    Focal Loss untuk menangani class imbalance secara lebih agresif.
    FL(pt) = -alpha_t * (1 - pt)^gamma * log(pt)

    Alternatif CrossEntropyLoss + weights.
    Focal loss lebih efektif saat imbalance sangat ekstrem.
    """

    def __init__(self, weight: torch.Tensor = None, gamma: float = 2.0):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt      = torch.exp(-ce_loss)
        focal   = (1 - pt) ** self.gamma * ce_loss
        return focal.mean()


if USE_FOCAL_LOSS:
    criterion = FocalLoss(weight=cw_tensor, gamma=FOCAL_GAMMA)
    print(f"[OK] Menggunakan Focal Loss (gamma={FOCAL_GAMMA}) + class weights.")
else:
    criterion = nn.CrossEntropyLoss(weight=cw_tensor)
    print("[OK] Menggunakan CrossEntropyLoss + class weights.")

# Optimizer: hanya optimize parameter yang trainable
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=1e-4,
)

# LR Scheduler: Cosine annealing
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-6,
)

print(f"[OK] Optimizer: AdamW (lr={LEARNING_RATE}, wd=1e-4)")
print(f"[OK] Scheduler: CosineAnnealingLR (T_max={NUM_EPOCHS})")

---
## CELL 10 – Fungsi Training & Evaluasi

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device) -> tuple[float, float]:
    """Jalankan satu epoch training. Return (avg_loss, accuracy)."""
    model.train()
    total_loss = 0.0
    correct    = 0
    n_samples  = 0

    for batch in tqdm(loader, desc="  Train", leave=False, unit="batch"):
        pixel_values = batch['pixel_values'].to(device, non_blocking=True)
        labels       = batch['labels'].to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(pixel_values)
        loss   = criterion(logits, labels)
        loss.backward()
        # Gradient clipping untuk stabilitas
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds      = logits.argmax(dim=-1)
        correct    += (preds == labels).sum().item()
        n_samples  += labels.size(0)

    return total_loss / n_samples, correct / n_samples


@torch.no_grad()
def evaluate(model, loader, criterion, device) -> tuple[float, float]:
    """Evaluasi model pada loader. Return (avg_loss, accuracy)."""
    model.eval()
    total_loss = 0.0
    correct    = 0
    n_samples  = 0

    for batch in tqdm(loader, desc="  Eval", leave=False, unit="batch"):
        pixel_values = batch['pixel_values'].to(device, non_blocking=True)
        labels       = batch['labels'].to(device, non_blocking=True)

        logits = model(pixel_values)
        loss   = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        preds       = logits.argmax(dim=-1)
        correct    += (preds == labels).sum().item()
        n_samples  += labels.size(0)

    return total_loss / n_samples, correct / n_samples


def save_checkpoint(model, optimizer, scheduler, epoch, val_loss, checkpoint_dir):
    """Simpan checkpoint ke Google Drive."""
    path = os.path.join(checkpoint_dir, f"checkpoint_epoch{epoch:03d}.pt")
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'val_loss': val_loss,
    }, path)
    print(f"  [Checkpoint] Disimpan: {path}")


def load_checkpoint(model, optimizer, scheduler, checkpoint_path: str) -> int:
    """
    Muat checkpoint untuk melanjutkan training setelah disconnect.
    Return epoch terakhir yang sudah disimpan.
    """
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    print(f"[OK] Checkpoint dimuat: epoch={ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f}")
    return ckpt['epoch']


print("[OK] Fungsi training dan evaluasi berhasil didefinisikan.")

---
## CELL 11 – Training Loop (Kerangka Dasar)

In [ ]:
# ============================================================
# RESUME DARI CHECKPOINT (opsional)
# Jika sesi Colab disconnect, isi RESUME_CHECKPOINT dengan path
# file checkpoint terakhir, misal:
#   RESUME_CHECKPOINT = os.path.join(CHECKPOINT_DIR, "checkpoint_epoch010.pt")
# ============================================================
RESUME_CHECKPOINT = None   # ganti ke path checkpoint jika ingin resume

start_epoch = 0
if RESUME_CHECKPOINT and os.path.exists(RESUME_CHECKPOINT):
    start_epoch = load_checkpoint(model, optimizer, scheduler, RESUME_CHECKPOINT)
    print(f"[INFO] Melanjutkan training dari epoch {start_epoch + 1}")
else:
    print(f"[INFO] Mulai training dari epoch 1")

# ── Training Loop ──────────────────────────────────────────
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_loss = float('inf')

print(f"\nTraining: {NUM_EPOCHS} epoch, batch={BATCH_SIZE}, freeze={FREEZE_BACKBONE}")
print(f"Device: {device}")
print("="*60)

for epoch in range(start_epoch, NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")

    # Train
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    # Validasi
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    # Update scheduler
    scheduler.step()

    # Catat history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
    print(f"  Val   Loss: {val_loss:.4f}   | Val   Acc: {val_acc*100:.2f}%")
    print(f"  LR        : {scheduler.get_last_lr()[0]:.6f}")

    # Simpan checkpoint periodik ke Google Drive
    # Ini penting agar progress tidak hilang jika Colab disconnect
    if (epoch + 1) % SAVE_EVERY_N_EPOCHS == 0:
        save_checkpoint(model, optimizer, scheduler, epoch + 1, val_loss, CHECKPOINT_DIR)

    # Simpan model terbaik
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
        torch.save(model.state_dict(), best_model_path)
        print(f"  ★ Best model disimpan (val_loss={val_loss:.4f})")

print("\n[OK] Training selesai!")
print(f"  Best val_loss: {best_val_loss:.4f}")

---
## CELL 12 – Evaluasi Final di Test Set

In [ ]:
# Muat model terbaik dan evaluasi di test set
best_model_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    print(f"[OK] Model terbaik dimuat dari: {best_model_path}")

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"\n=== EVALUASI TEST SET ===")
print(f"  Test Loss    : {test_loss:.4f}")
print(f"  Test Accuracy: {test_acc*100:.2f}%")

In [ ]:
# Classification report per kelas
from sklearn.metrics import classification_report, confusion_matrix

all_preds  = []
all_labels = []

model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Prediksi test", unit="batch"):
        pv     = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)
        logits = model(pv)
        preds  = logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

label_names = [IDX_TO_LABEL[i] for i in range(NUM_CLASSES)]
print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(all_labels, all_preds, target_names=label_names))

print("=== CONFUSION MATRIX ===")
cm = confusion_matrix(all_labels, all_preds)
cm_df = pd.DataFrame(cm, index=label_names, columns=label_names)
print(cm_df)